# B1.16 · Attesting control intent for agents and MCP servers

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline**  ·  *Security of AI*

Builds on **[B1.15 · Securing the developers' coding agents](https://spbreed.github.io/cyber-commons/lessons/B1.15.html)**.

| | |
|---|---|
| Open-source tooling | in-toto, Sigstore, OSCAL, OPA |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

"We enforce least privilege" is true of some deployment, at some time, and nothing binds it to the system running right now. An attestation is the binding — and the discipline is that it must refuse to say more than it can show.

## 2 · The framework

```
   claim                          attestation
   "we enforce least privilege"   subject: deployment_id @ digest
            |                     predicate: per-control verdicts + evidence
        unbound                   signed, re-issuable on any change

   the ceiling that keeps it honest
   +------------------------------------------------+
   | INTENT_EVIDENCED   strongest static verdict     |
   | PARTIAL (capped)   sandbox egress, injection    |
   | PASS               not in the vocabulary        |
   +------------------------------------------------+
```

Every control claim in this pipeline has the same weakness: it is a sentence in
a document, and nothing binds it to a running system. "We enforce least
privilege" is true of some deployment, at some time, and there is no way to
re-check it when the image, the role or the tool surface changes.

An **attestation** fixes the binding. It is a signed statement about a specific
deployment, carrying per-control verdicts and the evidence behind each one, that
can be re-issued whenever anything it describes changes.

Two design decisions carry the whole idea.

**A `deployment_id` is the join key.** It resolves to a manifest of
content-addressed artefacts — repo at a commit, image by digest, IAM role,
workload identity, gateway route, guardrail ID, downstream services. Without it
an IAM finding, an identity entry and a gateway policy are three unrelated facts
about three things that may not be the same system.

**Eleven skills, not one.** One resolver, nine collectors split along
evidence-source boundaries (code, IAM, network, sandbox, identity, gateway,
ingestion, risk register, entitlements), and one signer. They split there
because each needs different API clients — and they stay separate because a
single mega-skill produces context bloat and verdicts nobody can read.

Then the part that makes it honest. **Not every control is equally verifiable:**

| Control | Confidence | Why |
|---|---|---|
| C1 default-deny / least privilege | HIGH | policy documents plus observed usage are readable |
| C3 identity chain / OBO | HIGH | delegation is impossible without an actor token, so its presence is proof |
| C4 gateway routing | HIGH **if** egress is enforced below the application | otherwise an agent opens a socket and bypasses it |
| C2 sandbox / no egress | **PARTIAL, capped** | absence of a covert channel is not provable; DNS and object-storage bypasses are documented |
| C5 injection screening | **PARTIAL, capped** | detector presence is verifiable; adaptive attacks drive published defences back above 95% success |

A tool that reports PASS on C2 or C5 is wrong, and the cap belongs in the
artefact rather than in a footnote.

## 3 · The skill that does the static half

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/attestation/agent-code-surface-analyzer/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: agent-code-surface-analyzer
description: >-
  Statically enumerate an agent or MCP server's declared tools, dangerous
  actions and declared-versus-actual capability surface from its repository
  or image. Use to inventory what a deployment can do before trusting what
  it says it does, to check MCP tool annotations against the code, or to
  baseline tool descriptions for rug-pull detection.
allowed-tools: Read, Grep, Glob, Bash
---

# Agent Code Surface Analyzer

**Controls:** Static basis for controls 1, 2 and 5

## What this establishes

Every other control is about constraining capability. This skill establishes
what the capability actually **is**, from the code rather than from the
manifest — because the manifest is a claim made by the thing being audited.

## Procedure

1. **Enumerate declared tools.** From the MCP tool manifest, the tool registry,
   or the decorator/registration sites in code.

2. **Classify each tool by what it actually reaches**, by locating sinks:
   - `subprocess`, `os.system`, `exec`, `eval` → **process execution**
   - `open(...,'w')`, file writes, `shutil`, `os.remove` → **filesystem write**
   - HTTP clients, sockets → **network egress**
   - SDK credential reads, environment access → **credential access**
   - `DELETE`, `DROP`, `TRUNCATE` → **destructive downstream**

3. **Cross-check annotations against the code.** MCP tool annotations
   (`readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint`) are
   **hints, not guarantees** — the specification says so explicitly, and a
   server can mislabel a destructive tool as read-only. Treat every annotation
   as a claim to verify, never as truth.

   Apply the spec's pessimistic defaults: an **unannotated** tool is assumed
   `destructiveHint: true` and `openWorldHint: true`.

4. **Record the declared-versus-actual delta.** A tool annotated `readOnlyHint`
   whose body writes a file is the highest-value finding this skill produces.

5. **Hash the tool descriptions.** Store `tool_description_hash` per tool. This
   is the rug-pull baseline: a server can change a tool's description after the
   client approved it, and the hash is what detects that.

## Output contract

```json
{
  "deployment_id": "str",
  "tools": [
    {"name": "str", "capability_class": ["process|filesystem|network|credential|destructive"],
     "annotations": {"readOnlyHint": false, "destructiveHint": true,
                     "idempotentHint": false, "openWorldHint": true},
     "annotations_present": true,
     "declared_vs_actual": "match|understated|overstated",
     "evidence": [{"file": "str", "line": 0, "sink": "str"}],
     "tool_description_hash": "str"}
  ],
  "dangerous_actions": ["str"],
  "unannotated_count": 0,
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Believing the annotations.** They are advisory. Cross-check or do not
  report on them at all.
- **Missing dynamic registration.** Tools registered at runtime from config do
  not appear in a static scan; record that as a `PARTIAL`, not a clean pass.
- **Hashing the tool name instead of the description.** The description is what
  the model reads and what a rug-pull changes.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## 4 · Control intent, and why it is the honest static claim

Static analysis cannot show that a control **holds**. It can show that somebody **intended** it — an imported sandbox, a validated audience claim, a provenance tag. That is a smaller claim and a true one, so the analyser never emits PASS.

In [ ]:
SIGNALS = {
 "C1_default_deny_least_privilege": ["default_deny", "allowlist", "policy engine",
                                     "authorisation check"],
 "C2_sandbox_no_egress":            ["isolation runtime", "kernel confinement",
                                     "network mode control"],
 "C3_identity_chain_obo":           ["workload identity", "delegation claim",
                                     "audience validation", "token exchange"],
 "C4_gateway_guardrails":           ["gateway", "guardrail", "egress policy"],
 "C5_injection_screening":          ["injection detector", "sanitisation",
                                     "provenance tagging"],
}
CEILINGS = {
 "C2_sandbox_no_egress": "absence of a covert channel is not provable from source",
 "C5_injection_screening": "detector presence is verifiable; robustness is not",
}
RUNTIME_ONLY = {
 "C1_default_deny_least_privilege": "observed usage and the effective role policy",
 "C4_gateway_guardrails": "reachability testing from the deployment network",
}

def verdict(control, hits):
    if not hits:
        return "NO_INTENT_FOUND", "no signal for this control in the source"
    if control in CEILINGS:
        return "PARTIAL", CEILINGS[control]
    if control in RUNTIME_ONLY:
        return "INTENT_EVIDENCED", f"runtime verdict needs {RUNTIME_ONLY[control]}"
    return "INTENT_EVIDENCED", f"{len(hits)} signals; enforcement not shown"

for c in sorted(SIGNALS):
    v, why = verdict(c, SIGNALS[c])
    print(f"{c:34s}{v:18s}{why[:44]}")
print()
print("PASS is not in the vocabulary. The strongest static verdict is")
print("INTENT_EVIDENCED, and two controls cannot exceed PARTIAL at all.")
assert "PASS" not in {verdict(c, SIGNALS[c])[0] for c in SIGNALS}

## 5 · Run against ten real repositories

These are the verdicts the analyser in `labs/attestation/control_intent.py` produced against the five most-deployed open-source MCP repositories and five most-used agent frameworks, cloned at HEAD. Not a simulation — the counts below are what the scan returned.

In [ ]:
CORPUS = [
 {
  "repo": "awslabs_mcp",
  "kind": "mcp",
  "files": 2616,
  "tool_sites": 140,
  "sinks": 5,
  "annotated": True,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "crewAIInc_crewAI",
  "kind": "agent",
  "files": 2105,
  "tool_sites": 35,
  "sinks": 5,
  "annotated": False,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "github_github-mcp-server",
  "kind": "mcp",
  "files": 258,
  "tool_sites": 41,
  "sinks": 3,
  "annotated": False,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "langchain-ai_langchain",
  "kind": "agent",
  "files": 2673,
  "tool_sites": 99,
  "sinks": 5,
  "annotated": False,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "langchain-ai_langgraph",
  "kind": "agent",
  "files": 538,
  "tool_sites": 20,
  "sinks": 4,
  "annotated": False,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "NO_INTENT_FOUND",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "microsoft_autogen",
  "kind": "agent",
  "files": 707,
  "tool_sites": 49,
  "sinks": 5,
  "annotated": True,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "modelcontextprotocol_python-sdk",
  "kind": "mcp",
  "files": 894,
  "tool_sites": 710,
  "sinks": 4,
  "annotated": True,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "modelcontextprotocol_servers",
  "kind": "mcp",
  "files": 100,
  "tool_sites": 9,
  "sinks": 3,
  "annotated": True,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "NO_INTENT_FOUND",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "NO_INTENT_FOUND"
  }
 },
 {
  "repo": "modelcontextprotocol_typescript-sdk",
  "kind": "mcp",
  "files": 957,
  "tool_sites": 58,
  "sinks": 2,
  "annotated": True,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "NO_INTENT_FOUND",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 },
 {
  "repo": "openai_openai-agents-python",
  "kind": "agent",
  "files": 998,
  "tool_sites": 182,
  "sinks": 5,
  "annotated": False,
  "verdicts": {
   "C1": "INTENT_EVIDENCED",
   "C2": "PARTIAL",
   "C3": "INTENT_EVIDENCED",
   "C4": "INTENT_EVIDENCED",
   "C5": "PARTIAL"
  }
 }
]

print(f"{'repository':36s}{'kind':7s}{'files':>6}{'tools':>7}{'sinks':>6}  C1   C2   C3   C4   C5")
SHORT = {"INTENT_EVIDENCED": "INT", "PARTIAL": "PART", "NO_INTENT_FOUND": "-"}
for r in CORPUS:
    v = "".join(f"{SHORT[r['verdicts'][c]]:>5}" for c in ("C1","C2","C3","C4","C5"))
    print(f"{r['repo']:36s}{r['kind']:7s}{r['files']:>6}{r['tool_sites']:>7}{r['sinks']:>6}{v}")

evals = [r["verdicts"][c] for r in CORPUS for c in ("C1","C2","C3","C4","C5")]
print(f"\ncontrol evaluations : {len(evals)}")
for k in ("INTENT_EVIDENCED", "PARTIAL", "NO_INTENT_FOUND"):
    print(f"   {k:20s}{evals.count(k)}")
print(f"   {'PASS':20s}{evals.count('PASS')}   <- static evidence cannot prove enforcement")
assert evals.count("PASS") == 0

## 6 · What the scan actually found

Three findings worth more than the table.

In [ ]:
unannotated = [r for r in CORPUS if r["kind"] == "mcp" and not r["annotated"]]
print("1. MCP servers shipping NO tool annotations:")
for r in unannotated:
    print(f"   {r['repo']:36s}{r['tool_sites']} tool declaration sites")
print("   The specification is explicit that annotations are hints, not")
print("   guarantees - and that an UNANNOTATED tool must be assumed")
print("   destructiveHint=true and openWorldHint=true. Every one of those")
print("   tool sites inherits that pessimistic default.")

gaps = [(r["repo"], c) for r in CORPUS for c in ("C1","C2","C3","C4","C5")
        if r["verdicts"][c] == "NO_INTENT_FOUND"]
print(f"\n2. Controls with no intent anywhere in the source: {len(gaps)}")
for repo, c in gaps:
    print(f"   {repo:36s}{c}")

capped = [c for r in CORPUS for c in ("C2","C5") if r["verdicts"][c] == "PARTIAL"]
print(f"\n3. Verdicts capped at PARTIAL by the ceiling rule: {len(capped)}")
print("   Not because the evidence was weak - because the claim is not")
print("   provable. An attestation that reported PASS here would be a")
print("   signed, tamper-evident overstatement, which is worse than none.")
assert unannotated and gaps and capped

## 7 · The artefact

An in-toto statement, subject-bound to the deployment, predicate in assessment-results vocabulary. The signer is a separate skill and a separate role — an attester that also decides whether it passed is not an attestation.

In [ ]:
import json

def attestation(repo_row, commit="c9e71f7"):
    controls = []
    for c in ("C1","C2","C3","C4","C5"):
        full = [k for k in SIGNALS if k.startswith(c)][0]
        controls.append({
            "id": full,
            "verdict": repo_row["verdicts"][c],
            "confidence": "CAPPED" if full in CEILINGS else "STATIC",
            "evidence": [{"skill": "agent-code-surface-analyzer",
                          "uri": f"labs/attestation/oss-corpus-results.json#{repo_row['repo']}"}],
        })
    return {
      "_type": "https://in-toto.io/Statement/v1",
      "subject": [{"name": repo_row["repo"], "digest": {"gitCommit": commit}}],
      "predicateType": "https://cyber-commons/attestations/ai-control-intent/v1",
      "predicate": {"deployment_id": repo_row["repo"],
                    "scope": "static source analysis only",
                    "controls": controls,
                    "drift": {"since": None, "changed": []}},
      "signatures": [],
    }

a = attestation(CORPUS[0])
print(json.dumps(a, indent=1)[:600])
print("   ...")
print()
print("`signatures` is empty and says so. A relying party MUST fail closed on a")
print("missing or unsigned attestation: a signed file can be deleted, and")
print("absence must never be read as a pass.")
assert a["signatures"] == [] and a["predicate"]["scope"].startswith("static")

## What you just proved

Five controls resolve to INTENT_EVIDENCED, PARTIAL or NO_INTENT_FOUND and never to PASS. Across ten real repositories and fifty control evaluations the analyser returns 30 INTENT_EVIDENCED, 16 PARTIAL, 4 NO_INTENT_FOUND and zero PASS — with one widely-deployed MCP server shipping no tool annotations at all, so all of its tool sites inherit the specification's destructive, open-world default.

## Your turn

Run the analyser against one agent or MCP server you actually deploy. The interesting output is not the verdicts — it is the controls that come back NO_INTENT_FOUND, because those are the ones nobody has started.

---

**Next → [B1.17 · Bonus — Google Mantis, the pipeline in production](https://spbreed.github.io/cyber-commons/lessons/B1.17.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.16.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.16.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*